In [14]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier

In [15]:
df=pd.read_csv("data/heart.csv")
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [16]:
x=df.drop(columns=["HeartDisease"])
y=df["HeartDisease"]

In [17]:
num_cols=x.select_dtypes(include=["int64","float64"]).columns
cat_cols=x.select_dtypes(include=["object"]).columns

preprocessor=ColumnTransformer(
    transformers=[
        ("num",StandardScaler(),num_cols),
        ("cat",OneHotEncoder(drop="first",handle_unknown="ignore"),cat_cols)
    ]
)

/var/folders/8j/5h3gc34970x27yw4ybhhgzfc0000gn/T/ipykernel_21958/2539957870.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols=x.select_dtypes(include=["object"]).columns


In [18]:
# light GBM
lgbm_pipeline=Pipeline(
    steps=[
        ("preprocessing",preprocessor),
        ("classifier",LGBMClassifier(random_state=42))
    ]
)


In [19]:
import numpy as np

param_distributions={
    "classifier__n_estimators":[50,100,200],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__max_depth': [3, 5, 7],
    'classifier__subsample': [0.8, 1.0]
}

In [20]:
from sklearn.model_selection import RandomizedSearchCV

random_search=RandomizedSearchCV(
    estimator=lgbm_pipeline,
    param_distributions=param_distributions,
    n_iter=10,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

random_search.fit(x,y)

print("best accuracy :",random_search.best_score_)
print("optimmal settings",random_search.best_params_)

[LightGBM] [Info] Number of positive: 406, number of negative: 328
[LightGBM] [Info] Number of positive: 407, number of negative: 328[LightGBM] [Info] Number of positive: 406, number of negative: 328

[LightGBM] [Info] Number of positive: 406, number of negative: 328
[LightGBM] [Info] Number of positive: 406, number of negative: 328
[LightGBM] [Info] Number of positive: 407, number of negative: 328
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001881 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001837 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 366
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001892 seconds.
You can s